### Instalação

In [ ]:
!apt-get install libffi-dev
!pip install PySUS==0.15.0
!pip install pandas

### Download dos dados

In [4]:
import urllib.request

# Definicao de strings reutilizaveis
base_url = 'ftp://ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/FINAIS/DENGBR'
base_fname = 'DENGBR'
dbc = '.dbc'

# Download dos dados entre 2014 e 2023
for i in range(14, 24):
    urllib.request.urlretrieve(base_url + str(i) + dbc, base_fname + str(i) + dbc)
    print('Downloaded DENGBR' + str(i) + dbc)

Downloaded DENGBR14.dbc


### Filtros

In [5]:
import pandas as pd
from pysus.data.local import Data

path = "/caminho/do/arquivo.dbc"
file = Data(path)

df = file.to_dataframe()

In [11]:
# ler as colunas DT_NOTIFIC, ID_MUNICIP, ID_UNIDADE, SEM_NOT, CLASSI_FIN
selected_cols = ["DT_NOTIFIC","ID_MUNICIP","ID_UNIDADE","SEM_NOT","CLASSI_FIN"]

df1 = df[selected_cols]

# filtra os casos confirmados 10 - dengue e 12 - dengue grave
confirmados = df1[(df1['CLASSI_FIN'] == '10') | (df1['CLASSI_FIN'] == '12')]

# filtra os casos do municipio do RJ
confirmados_rj = confirmados[confirmados['ID_MUNICIP'] == '330455']

### Join e conversão

In [14]:
# ler as colunas CO_CNES, NU_LONGITUDE e NU_LATITUDE
cols = ["CO_CNES", "NU_LONGITUDE", "NU_LATITUDE"]

estab = pd.read_csv("/content/tbEstabelecimento202603.csv", sep=";", encoding="latin1", dtype=str, usecols=cols)

# renomeia colunas e remove colunas nulas
casos_georef = confirmados_rj.set_index("ID_UNIDADE").join(estab.set_index("CO_CNES"))
casos_georef.rename(columns={"NU_LONGITUDE" : "lng", "NU_LATITUDE" : "lat"}, inplace=True)
casos_georef = casos_georef.dropna()

# formação do csv final
cols = ["lng", "lat"]
casos_georef = casos_georef[cols]

#ano = 2023 (substituir pelo ano do arquivo)
nome_arquivo = "casos_georef_" + str(ano) + ".csv"
casos_georef.to_csv(nome_arquivo, sep=",", index=False)

from google.colab import files
files.download(nome_arquivo)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>